# Lab 16 — ResNet from Scratch
BasicBlock, BottleNeck and custom ResNet builders. Streamlined Colab edition.

In [ ]:
import torch,torch.nn as nn
class BasicBlock(nn.Module):
 expansion=1
 def __init__(self,inc,outc,stride=1):
  super().__init__(); self.f=nn.Sequential(nn.Conv2d(inc,outc,3,stride,1,bias=False),nn.BatchNorm2d(outc),nn.ReLU(),nn.Conv2d(outc,outc,3,1,1,bias=False),nn.BatchNorm2d(outc)); self.s=nn.Identity() if stride==1 and inc==outc else nn.Sequential(nn.Conv2d(inc,outc,1,stride,bias=False),nn.BatchNorm2d(outc)); self.r=nn.ReLU()
 def forward(self,x): return self.r(self.f(x)+self.s(x))
class BottleNeck(nn.Module):
 expansion=4
 def __init__(self,inc,outc,stride=1):
  super().__init__(); e=outc*4; self.f=nn.Sequential(nn.Conv2d(inc,outc,1,bias=False),nn.BatchNorm2d(outc),nn.ReLU(),nn.Conv2d(outc,outc,3,stride,1,bias=False),nn.BatchNorm2d(outc),nn.ReLU(),nn.Conv2d(outc,e,1,bias=False),nn.BatchNorm2d(e)); self.s=nn.Identity() if stride==1 and inc==e else nn.Sequential(nn.Conv2d(inc,e,1,stride,bias=False),nn.BatchNorm2d(e)); self.r=nn.ReLU()
 def forward(self,x): return self.r(self.f(x)+self.s(x))

In [ ]:
class ResNet(nn.Module):
 def __init__(self,block,nums,classes=9):
  super().__init__(); self.c=64; self.stem=nn.Sequential(nn.Conv2d(3,64,3,1,1,bias=False),nn.BatchNorm2d(64),nn.ReLU()); self.l1=self.make(block,64,nums[0],1); self.l2=self.make(block,128,nums[1],2); self.l3=self.make(block,256,nums[2],2); self.l4=self.make(block,512,nums[3],2); self.pool=nn.AdaptiveAvgPool2d(1); self.fc=nn.Linear(512*block.expansion,classes)
 def make(self,b,o,n,s):
  layers=[b(self.c,o,s)]; self.c=o*b.expansion
  for _ in range(1,n): layers.append(b(self.c,o))
  return nn.Sequential(*layers)
 def forward(self,x): x=self.stem(x); x=self.l1(x); x=self.l2(x); x=self.l3(x); x=self.l4(x); return self.fc(self.pool(x).flatten(1))
def resnet18(c=9): return ResNet(BasicBlock,[2,2,2,2],c)
def resnet34(c=9): return ResNet(BasicBlock,[3,4,6,3],c)
def resnet50(c=9): return ResNet(BottleNeck,[3,4,6,3],c)
def resnet101(c=9): return ResNet(BottleNeck,[3,4,23,3],c)
def resnet152(c=9): return ResNet(BottleNeck,[3,8,36,3],c)
for fn in [resnet18,resnet34,resnet50]:
 m=fn(); print(fn.__name__,sum(p.numel() for p in m.parameters()),m(torch.randn(1,3,32,32)).shape)